In [ ]:
# -*- coding: utf-8 -*-
"""2D U-Net with Porosity, Specific Surface Area Analysis, and Performance Tracking"""

# Install required packages
!pip install torch torchvision torchaudio --quiet
!pip install tifffile scikit-image scipy matplotlib pandas --quiet
!pip install psutil gputil --quiet

import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
from skimage.filters import threshold_otsu, threshold_multiotsu
from scipy import stats
from scipy.ndimage import binary_opening, binary_closing
import pandas as pd
from datetime import datetime
import time
import psutil
import warnings
warnings.filterwarnings('ignore')

from tifffile import imread, imwrite

# Try to import GPUtil for GPU monitoring
try:
    import GPUtil
    GPU_AVAILABLE = True
except:
    GPU_AVAILABLE = False
    print("⚠️ GPUtil not available, GPU memory tracking will be limited")

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Your specific paths
NOISY_PATH = '/content/drive/MyDrive/Dataset TA/Libo/No RA Removal/512/Libo 512 RA.tif'
CLEAN_PATH = '/content/drive/MyDrive/Dataset TA/Libo/RA Removal 13/512/Libo 512.tif'

# Verify files
print(f"Noisy file exists: {os.path.exists(NOISY_PATH)}")
print(f"Clean file exists: {os.path.exists(CLEAN_PATH)}")

if os.path.exists(NOISY_PATH) and os.path.exists(CLEAN_PATH):
    print(f"\n✓ Both files found!")
    print(f"  Noisy: {os.path.getsize(NOISY_PATH)/1e6:.1f} MB")
    print(f"  Clean: {os.path.getsize(CLEAN_PATH)/1e6:.1f} MB")

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\nUsing device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory Available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# Configuration
class Config:
    NOISY_PATH = NOISY_PATH
    CLEAN_PATH = CLEAN_PATH

    PATCH_SIZE = 128
    PATCHES_PER_SLICE = 10
    BATCH_SIZE = 128
    EPOCHS = 100
    LEARNING_RATE = 8e-4

    PATIENCE = 15
    LR_PATIENCE = 5
    LR_FACTOR = 0.5

    VOXEL_SIZE_UM = 5.22  # Ukuran voxel dalam mikrometer (μm)

    # BIT DEPTH CONFIGURATION - PENTING UNTUK APPLE-TO-APPLE COMPARISON
    TARGET_BIT_DEPTH = 8  # 8-bit (0-255) atau 16-bit (0-65535) atau 32-bit (0.0-1.0)
    # Untuk konsistensi, kita gunakan 8-bit untuk semua analisis
    NORMALIZE_TO_UINT8 = True  # Convert semua ke uint8 untuk fair comparison

config = Config()

print("\n" + "="*70)
print("2D U-NET CONFIGURATION")
print("="*70)
print(f"  Patch Size: {config.PATCH_SIZE}×{config.PATCH_SIZE}")
print(f"  Patches per Slice: {config.PATCHES_PER_SLICE}")
print(f"  Batch Size: {config.BATCH_SIZE}")
print(f"  Max Epochs: {config.EPOCHS}")
print(f"  Learning Rate: {config.LEARNING_RATE}")
print(f"\n  Voxel Size: {config.VOXEL_SIZE_UM} μm")
print(f"\n  🔧 BIT DEPTH CONFIGURATION:")
print(f"  Target Bit Depth: {config.TARGET_BIT_DEPTH}-bit")
print(f"  Normalize to uint8: {config.NORMALIZE_TO_UINT8}")
print(f"  → All comparisons will be in {config.TARGET_BIT_DEPTH}-bit space for fair analysis")

# ============================================================================
# PERFORMANCE TRACKER WITH HARDWARE MONITORING
# ============================================================================

class HardwareMonitor:
    """Monitor CPU, RAM, and GPU usage"""
    def __init__(self):
        self.process = psutil.Process()
        self.gpu_available = torch.cuda.is_available()
        self.gputil_available = GPU_AVAILABLE

    def get_cpu_usage(self):
        """Get current CPU usage percentage"""
        return psutil.cpu_percent(interval=0.1)

    def get_ram_usage(self):
        """Get current RAM usage in GB and percentage"""
        ram = psutil.virtual_memory()
        return {
            'used_gb': ram.used / (1024**3),
            'total_gb': ram.total / (1024**3),
            'percent': ram.percent,
            'available_gb': ram.available / (1024**3)
        }

    def get_process_ram(self):
        """Get RAM usage by current process"""
        mem_info = self.process.memory_info()
        return mem_info.rss / (1024**3)  # GB

    def get_gpu_usage(self):
        """Get GPU memory usage"""
        if not self.gpu_available:
            return None

        gpu_stats = {
            'allocated_gb': torch.cuda.memory_allocated() / (1024**3),
            'reserved_gb': torch.cuda.memory_reserved() / (1024**3),
            'max_allocated_gb': torch.cuda.max_memory_allocated() / (1024**3),
        }

        # Try to get more detailed GPU info using GPUtil
        if self.gputil_available:
            try:
                gpus = GPUtil.getGPUs()
                if gpus:
                    gpu = gpus[0]
                    gpu_stats.update({
                        'total_gb': gpu.memoryTotal / 1024,
                        'used_gb': gpu.memoryUsed / 1024,
                        'free_gb': gpu.memoryFree / 1024,
                        'utilization_percent': gpu.memoryUtil * 100,
                        'gpu_load_percent': gpu.load * 100,
                        'temperature_c': gpu.temperature
                    })
            except:
                pass

        return gpu_stats

    def get_all_stats(self):
        """Get all hardware statistics"""
        stats = {
            'cpu_percent': self.get_cpu_usage(),
            'ram': self.get_ram_usage(),
            'process_ram_gb': self.get_process_ram(),
        }

        if self.gpu_available:
            stats['gpu'] = self.get_gpu_usage()

        return stats

    def print_stats(self, label="Current"):
        """Print formatted hardware stats"""
        stats = self.get_all_stats()

        print(f"\n{'='*70}")
        print(f"{label} Hardware Usage")
        print(f"{'='*70}")
        print(f"  CPU Usage: {stats['cpu_percent']:.1f}%")
        print(f"  RAM: {stats['ram']['used_gb']:.2f} / {stats['ram']['total_gb']:.2f} GB ({stats['ram']['percent']:.1f}%)")
        print(f"  Process RAM: {stats['process_ram_gb']:.2f} GB")

        if 'gpu' in stats and stats['gpu']:
            gpu = stats['gpu']
            print(f"\n  GPU Memory:")
            print(f"    Allocated: {gpu['allocated_gb']:.2f} GB")
            print(f"    Reserved:  {gpu['reserved_gb']:.2f} GB")
            print(f"    Peak:      {gpu['max_allocated_gb']:.2f} GB")

            if 'total_gb' in gpu:
                print(f"    Total:     {gpu['total_gb']:.2f} GB")
                print(f"    Used:      {gpu['used_gb']:.2f} GB ({gpu['utilization_percent']:.1f}%)")
                print(f"    GPU Load:  {gpu['gpu_load_percent']:.1f}%")
                if 'temperature_c' in gpu:
                    print(f"    Temp:      {gpu['temperature_c']:.1f}°C")


class PerformanceTracker:
    """Track execution time and hardware usage for different phases"""
    def __init__(self):
        self.times = {}
        self.hardware_stats = {}
        self.current_phase = None
        self.start_time = None
        self.hw_monitor = HardwareMonitor()

        # Store initial state
        self.initial_stats = self.hw_monitor.get_all_stats()

    def start(self, phase_name):
        """Start timing a phase and record hardware state"""
        self.current_phase = phase_name
        self.start_time = time.time()

        # Record starting hardware state
        self.hardware_stats[phase_name] = {
            'start': self.hw_monitor.get_all_stats()
        }

        print(f"\n⏱️  Starting: {phase_name}...")

        # Reset peak GPU memory at start of each phase
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()

    def end(self):
        """End timing current phase and record final hardware state"""
        if self.current_phase and self.start_time:
            elapsed = time.time() - self.start_time
            self.times[self.current_phase] = elapsed

            # Record ending hardware state
            end_stats = self.hw_monitor.get_all_stats()
            self.hardware_stats[self.current_phase]['end'] = end_stats
            self.hardware_stats[self.current_phase]['duration'] = elapsed

            # Calculate peak GPU memory for this phase
            if torch.cuda.is_available():
                peak_mem = torch.cuda.max_memory_allocated() / (1024**3)
                self.hardware_stats[self.current_phase]['peak_gpu_gb'] = peak_mem

            print(f"✓ Completed: {self.current_phase} ({self._format_time(elapsed)})")

            self.current_phase = None
            self.start_time = None
            return elapsed
        return 0

    def _format_time(self, seconds):
        """Format seconds to human readable"""
        if seconds < 60:
            return f"{seconds:.2f}s"
        elif seconds < 3600:
            mins = seconds / 60
            return f"{mins:.2f}min"
        else:
            hours = seconds / 3600
            return f"{hours:.2f}h"

    def get_summary(self):
        """Get summary of all times"""
        total = sum(self.times.values())
        summary = {
            'individual': self.times.copy(),
            'total': total,
            'formatted': {k: self._format_time(v) for k, v in self.times.items()},
            'total_formatted': self._format_time(total),
            'hardware': self.hardware_stats
        }
        return summary

    def print_summary(self):
        """Print formatted summary with hardware stats"""
        print("\n" + "="*70)
        print("PERFORMANCE SUMMARY - TIME")
        print("="*70)
        for phase, duration in self.times.items():
            percentage = (duration / sum(self.times.values())) * 100
            print(f"  {phase:<40} {self._format_time(duration):>12} ({percentage:>5.1f}%)")
        print("-" * 70)
        print(f"  {'TOTAL':<40} {self._format_time(sum(self.times.values())):>12} (100.0%)")

        # Print hardware summary
        print("\n" + "="*70)
        print("PERFORMANCE SUMMARY - HARDWARE")
        print("="*70)

        # Overall stats
        if torch.cuda.is_available():
            print("\n  Peak GPU Memory Usage by Phase:")
            for phase, stats in self.hardware_stats.items():
                if 'peak_gpu_gb' in stats:
                    print(f"    {phase:<40} {stats['peak_gpu_gb']:>8.2f} GB")

        print("\n  RAM Usage Summary:")
        for phase, stats in self.hardware_stats.items():
            if 'start' in stats and 'end' in stats:
                start_ram = stats['start']['process_ram_gb']
                end_ram = stats['end']['process_ram_gb']
                delta = end_ram - start_ram
                print(f"    {phase:<40} {start_ram:>6.2f} → {end_ram:>6.2f} GB (Δ{delta:+.2f} GB)")

    def get_peak_memory_usage(self):
        """Get peak memory usage across all phases"""
        peak_stats = {
            'peak_gpu_gb': 0,
            'peak_ram_gb': 0,
            'peak_process_ram_gb': 0
        }

        for phase, stats in self.hardware_stats.items():
            if 'peak_gpu_gb' in stats:
                peak_stats['peak_gpu_gb'] = max(peak_stats['peak_gpu_gb'], stats['peak_gpu_gb'])

            if 'end' in stats:
                peak_stats['peak_ram_gb'] = max(peak_stats['peak_ram_gb'],
                                                stats['end']['ram']['used_gb'])
                peak_stats['peak_process_ram_gb'] = max(peak_stats['peak_process_ram_gb'],
                                                        stats['end']['process_ram_gb'])

        return peak_stats

# Initialize tracker and hardware monitor
perf_tracker = PerformanceTracker()
hw_monitor = HardwareMonitor()

# Show initial hardware state
hw_monitor.print_stats("Initial")

# ============================================================================
# DATASET & MODEL (Same as before)
# ============================================================================

class SliceDataset(Dataset):
    """Dataset for 2D slices from 3D volume"""
    def __init__(self, noisy_volume, clean_volume, slice_indices, patch_size=64, patches_per_slice=20):
        self.noisy_volume = noisy_volume
        self.clean_volume = clean_volume
        self.slice_indices = slice_indices
        self.patch_size = patch_size
        self.patches_per_slice = patches_per_slice

    def __len__(self):
        return len(self.slice_indices) * self.patches_per_slice

    def __getitem__(self, idx):
        slice_idx = self.slice_indices[idx // self.patches_per_slice]

        noisy_slice = self.noisy_volume[slice_idx].astype(np.float32)
        clean_slice = self.clean_volume[slice_idx].astype(np.float32)

        noisy_slice = (noisy_slice - noisy_slice.min()) / (noisy_slice.max() - noisy_slice.min() + 1e-8)
        clean_slice = (clean_slice - clean_slice.min()) / (clean_slice.max() - clean_slice.min() + 1e-8)

        h, w = noisy_slice.shape
        ps = self.patch_size

        if h >= ps and w >= ps:
            h_start = np.random.randint(0, h - ps + 1)
            w_start = np.random.randint(0, w - ps + 1)

            noisy_patch = noisy_slice[h_start:h_start+ps, w_start:w_start+ps]
            clean_patch = clean_slice[h_start:h_start+ps, w_start:w_start+ps]
        else:
            noisy_patch = noisy_slice
            clean_patch = clean_slice

            if h < ps or w < ps:
                pad_h = max(0, ps - h)
                pad_w = max(0, ps - w)
                pad_top = pad_h // 2
                pad_bottom = pad_h - pad_top
                pad_left = pad_w // 2
                pad_right = pad_w - pad_left

                noisy_patch = np.pad(noisy_patch, ((pad_top, pad_bottom), (pad_left, pad_right)), mode='reflect')
                clean_patch = np.pad(clean_patch, ((pad_top, pad_bottom), (pad_left, pad_right)), mode='reflect')

        noisy_patch = noisy_patch[:ps, :ps]
        clean_patch = clean_patch[:ps, :ps]

        noisy_patch = torch.from_numpy(noisy_patch[np.newaxis, ...])
        clean_patch = torch.from_numpy(clean_patch[np.newaxis, ...])

        return noisy_patch, clean_patch


class DoubleConv(nn.Module):
    """Double convolution block for 2D U-Net"""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)


class UNet2D(nn.Module):
    """2D U-Net architecture for image denoising"""
    def __init__(self, in_ch=1, out_ch=1):
        super().__init__()

        self.enc1 = DoubleConv(in_ch, 64)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = DoubleConv(64, 128)
        self.pool2 = nn.MaxPool2d(2)
        self.enc3 = DoubleConv(128, 256)
        self.pool3 = nn.MaxPool2d(2)
        self.enc4 = DoubleConv(256, 512)
        self.pool4 = nn.MaxPool2d(2)

        self.bottleneck = DoubleConv(512, 1024)

        self.up4 = nn.ConvTranspose2d(1024, 512, 2, stride=2)
        self.dec4 = DoubleConv(1024, 512)
        self.up3 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.dec3 = DoubleConv(512, 256)
        self.up2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec2 = DoubleConv(256, 128)
        self.up1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec1 = DoubleConv(128, 64)

        self.out = nn.Conv2d(64, out_ch, 1)

    def _crop_and_concat(self, upsampled, bypass):
        _, _, h_up, w_up = upsampled.shape
        _, _, h_by, w_by = bypass.shape

        if h_by != h_up or w_by != w_up:
            diff_h = h_by - h_up
            diff_w = w_by - w_up

            if diff_h > 0:
                crop_top = diff_h // 2
                crop_bottom = diff_h - crop_top
                bypass = bypass[:, :, crop_top:h_by-crop_bottom, :]
            elif diff_h < 0:
                pad_top = (-diff_h) // 2
                pad_bottom = (-diff_h) - pad_top
                bypass = nn.functional.pad(bypass, (0, 0, pad_top, pad_bottom))

            if diff_w > 0:
                crop_left = diff_w // 2
                crop_right = diff_w - crop_left
                bypass = bypass[:, :, :, crop_left:w_by-crop_right]
            elif diff_w < 0:
                pad_left = (-diff_w) // 2
                pad_right = (-diff_w) - pad_left
                bypass = nn.functional.pad(bypass, (pad_left, pad_right, 0, 0))

        return torch.cat([upsampled, bypass], dim=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        e4 = self.enc4(self.pool3(e3))

        b = self.bottleneck(self.pool4(e4))

        d4 = self.up4(b)
        d4 = self._crop_and_concat(d4, e4)
        d4 = self.dec4(d4)

        d3 = self.up3(d4)
        d3 = self._crop_and_concat(d3, e3)
        d3 = self.dec3(d3)

        d2 = self.up2(d3)
        d2 = self._crop_and_concat(d2, e2)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        d1 = self._crop_and_concat(d1, e1)
        d1 = self.dec1(d1)

        return self.out(d1)


def calculate_metrics(original, denoised, data_range=1.0):
    """
    Calculate image quality metrics
    PENTING: original dan denoised harus sudah dalam bit depth yang sama!
    """
    mse = np.mean((original.astype(np.float64) - denoised.astype(np.float64)) ** 2)

    if original.ndim == 2:
        psnr_val = psnr(original, denoised, data_range=data_range)
        ssim_val = ssim(original, denoised, data_range=data_range)
    else:
        psnr_vals = []
        ssim_vals = []
        for i in range(original.shape[0]):
            psnr_vals.append(psnr(original[i], denoised[i], data_range=data_range))
            ssim_vals.append(ssim(original[i], denoised[i], data_range=data_range))
        psnr_val = np.mean(psnr_vals)
        ssim_val = np.mean(ssim_vals)

    return {
        'MSE': mse,
        'PSNR': psnr_val,
        'SSIM': ssim_val
    }


def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0

    for noisy, clean in loader:
        noisy, clean = noisy.to(device), clean.to(device)

        optimizer.zero_grad()
        output = model(noisy)
        loss = criterion(output, clean)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for noisy, clean in loader:
            noisy, clean = noisy.to(device), clean.to(device)
            output = model(noisy)
            loss = criterion(output, clean)
            total_loss += loss.item()

    return total_loss / len(loader)


def denoise_volume(model, noisy_volume, device, batch_size=64):
    """Denoise entire volume slice-by-slice with padding for dimension compatibility"""
    model.eval()

    n_slices = noisy_volume.shape[0]
    original_h, original_w = noisy_volume.shape[1], noisy_volume.shape[2]

    pad_h = (16 - original_h % 16) % 16
    pad_w = (16 - original_w % 16) % 16

    pad_top = pad_h // 2
    pad_bottom = pad_h - pad_top
    pad_left = pad_w // 2
    pad_right = pad_w - pad_left

    denoised_volume = np.zeros_like(noisy_volume, dtype=np.float32)

    print(f"Denoising {n_slices} slices...")
    if pad_h > 0 or pad_w > 0:
        print(f"  Padding input: ({original_h}, {original_w}) -> ({original_h+pad_h}, {original_w+pad_w})")

    for i in range(0, n_slices, batch_size):
        end_idx = min(i + batch_size, n_slices)
        batch_slices = noisy_volume[i:end_idx]

        batch_min = batch_slices.min(axis=(1, 2), keepdims=True)
        batch_max = batch_slices.max(axis=(1, 2), keepdims=True)
        batch_norm = (batch_slices - batch_min) / (batch_max - batch_min + 1e-8)

        batch_tensor = torch.from_numpy(batch_norm[:, np.newaxis, ...]).to(device)

        if pad_h > 0 or pad_w > 0:
            batch_tensor = nn.functional.pad(batch_tensor, (pad_left, pad_right, pad_top, pad_bottom), mode='reflect')

        with torch.no_grad():
            denoised_batch = model(batch_tensor)

        if pad_h > 0 or pad_w > 0:
            denoised_batch = denoised_batch[:, :, pad_top:pad_top+original_h, pad_left:pad_left+original_w]

        denoised_batch = denoised_batch.cpu().numpy()[:, 0]
        denoised_batch = denoised_batch * (batch_max[:, 0, 0] - batch_min[:, 0, 0])[:, np.newaxis, np.newaxis] + batch_min[:, 0, 0][:, np.newaxis, np.newaxis]

        denoised_volume[i:end_idx] = denoised_batch

        if (i // batch_size + 1) % 10 == 0:
            print(f"  Processed {end_idx}/{n_slices} slices")

    return denoised_volume


# ============================================================================
# NORMALIZATION FUNCTIONS - UNTUK APPLE-TO-APPLE COMPARISON
# ============================================================================

def normalize_to_range(volume, target_min=0, target_max=1):
    """
    Normalize volume to specific range [target_min, target_max]
    Preserves relative intensity relationships
    """
    v_min = volume.min()
    v_max = volume.max()

    if v_max - v_min < 1e-8:
        return np.full_like(volume, target_min)

    normalized = (volume - v_min) / (v_max - v_min)
    normalized = normalized * (target_max - target_min) + target_min

    return normalized


def convert_to_uint8(volume):
    """
    Convert volume to uint8 (0-255) with proper scaling
    """
    # First normalize to 0-1
    normalized = normalize_to_range(volume, 0, 1)

    # Then scale to 0-255 and convert to uint8
    uint8_volume = (normalized * 255).astype(np.uint8)

    return uint8_volume


def convert_to_uint16(volume):
    """
    Convert volume to uint16 (0-65535) with proper scaling
    """
    normalized = normalize_to_range(volume, 0, 1)
    uint16_volume = (normalized * 65535).astype(np.uint16)
    return uint16_volume


def standardize_volumes_for_comparison(noisy, denoised, ground_truth, bit_depth=8):
    """
    Standardize all volumes to same bit depth for fair comparison

    Parameters:
    -----------
    noisy, denoised, ground_truth : ndarray
        Input volumes (any bit depth)
    bit_depth : int
        Target bit depth (8 or 16 or 32)

    Returns:
    --------
    standardized volumes in same bit depth
    """
    if bit_depth == 8:
        noisy_std = convert_to_uint8(noisy)
        denoised_std = convert_to_uint8(denoised)
        truth_std = convert_to_uint8(ground_truth)
        data_range = 255
        dtype_name = 'uint8'

    elif bit_depth == 16:
        noisy_std = convert_to_uint16(noisy)
        denoised_std = convert_to_uint16(denoised)
        truth_std = convert_to_uint16(ground_truth)
        data_range = 65535
        dtype_name = 'uint16'

    else:  # 32-bit float, normalized to 0-1
        noisy_std = normalize_to_range(noisy, 0, 1).astype(np.float32)
        denoised_std = normalize_to_range(denoised, 0, 1).astype(np.float32)
        truth_std = normalize_to_range(ground_truth, 0, 1).astype(np.float32)
        data_range = 1.0
        dtype_name = 'float32'

    print(f"\n✓ Volumes standardized to {bit_depth}-bit ({dtype_name}) for fair comparison")
    print(f"  Data range: {0} - {data_range}")
    print(f"  Noisy dtype: {noisy_std.dtype}, range: [{noisy_std.min()}, {noisy_std.max()}]")
    print(f"  Denoised dtype: {denoised_std.dtype}, range: [{denoised_std.min()}, {denoised_std.max()}]")
    print(f"  Ground truth dtype: {truth_std.dtype}, range: [{truth_std.min()}, {truth_std.max()}]")

    return noisy_std, denoised_std, truth_std, data_range

# ============================================================================
# POROSITY & SPECIFIC GRAVITY ANALYSIS
# ============================================================================

def calculate_porosity(volume, method='otsu', manual_threshold=None):
    """
    Calculate porosity from 3D volume
    INPUT harus sudah dalam format yang consistent (0-1 normalized atau uint8)
    """
    # Normalize to 0-1 if not already
    if volume.dtype == np.uint8:
        volume_norm = volume.astype(np.float32) / 255.0
    elif volume.dtype == np.uint16:
        volume_norm = volume.astype(np.float32) / 65535.0
    else:
        volume_norm = (volume - volume.min()) / (volume.max() - volume.min() + 1e-8)

    if method == 'otsu':
        threshold_value = threshold_otsu(volume_norm)
    elif method == 'multiotsu':
        thresholds = threshold_multiotsu(volume_norm, classes=3)
        threshold_value = thresholds[0]
    elif method == 'manual':
        if manual_threshold is None:
            raise ValueError("manual_threshold must be provided for manual method")
        threshold_value = manual_threshold
    else:
        raise ValueError(f"Unknown method: {method}")

    binary_volume = volume_norm < threshold_value
    porosity = np.sum(binary_volume) / binary_volume.size * 100

    return porosity, binary_volume, threshold_value



def calculate_specific_surface_area(binary_volume, voxel_size_um=1.0):
    """
    Calculate Specific Surface Area (SSA) dari binary volume

    SSA mengukur luas permukaan interface antara pori dan solid per unit volume.
    Metode: Marching Cubes untuk estimasi surface area dari binary segmentation.

    Parameters:
    -----------
    binary_volume : ndarray (bool or uint8)
        Binary 3D volume (True/1 = pore space, False/0 = solid)
    voxel_size_um : float
        Ukuran voxel dalam mikrometer (μm)

    Returns:
    --------
    ssa : float
        Specific Surface Area dalam μm⁻¹ (surface area per unit volume)
    surface_area_um2 : float
        Total surface area dalam μm²
    volume_um3 : float
        Total volume dalam μm³
    """
    from skimage import measure

    # Ensure binary format
    if binary_volume.dtype != bool:
        binary_volume = binary_volume.astype(bool)

    # Calculate surface area using marching cubes
    # We measure the interface between pore (True) and solid (False)
    try:
        # Marching cubes to get the surface mesh
        verts, faces, normals, values = measure.marching_cubes(
            binary_volume.astype(np.float32),
            level=0.5,
            spacing=(voxel_size_um, voxel_size_um, voxel_size_um)
        )

        # Calculate surface area from faces
        # Each face is a triangle, calculate area of all triangles
        v0 = verts[faces[:, 0]]
        v1 = verts[faces[:, 1]]
        v2 = verts[faces[:, 2]]

        # Cross product to get triangle areas
        cross = np.cross(v1 - v0, v2 - v0)
        areas = 0.5 * np.sqrt(np.sum(cross**2, axis=1))
        surface_area_um2 = np.sum(areas)

    except Exception as e:
        # Fallback: estimate using edge detection (simpler method)
        print(f"  Warning: Marching cubes failed ({e}), using edge-based estimation")

        # Count interface voxels (edges between pore and solid)
        # Simple method: count voxels with at least one different neighbor
        from scipy import ndimage

        # Create structure element for 6-connectivity
        struct = ndimage.generate_binary_structure(3, 1)

        # Dilate and find boundary
        dilated = ndimage.binary_dilation(binary_volume, structure=struct)
        boundary = dilated & ~binary_volume

        # Count boundary voxels
        n_boundary_voxels = np.sum(boundary)

        # Each boundary voxel contributes approximately voxel_size^2 surface area
        surface_area_um2 = n_boundary_voxels * (voxel_size_um ** 2)

    # Calculate volume
    depth, height, width = binary_volume.shape
    volume_um3 = depth * height * width * (voxel_size_um ** 3)

    # Calculate SSA (surface area per unit volume)
    ssa = surface_area_um2 / volume_um3 if volume_um3 > 0 else 0

    return ssa, surface_area_um2, volume_um3


def analyze_ssa_per_slice(binary_volume, voxel_size_um=1.0):
    """
    Calculate Specific Surface Area untuk setiap slice (2D analysis)

    Parameters:
    -----------
    binary_volume : ndarray
        3D binary volume (True = pore space, False = solid)
    voxel_size_um : float
        Ukuran voxel dalam mikrometer (μm)

    Returns:
    --------
    ssa_per_slice : list
        SSA untuk setiap slice (dalam μm⁻¹)
    """
    from skimage import measure

    n_slices = binary_volume.shape[0]
    ssa_per_slice = []

    for i in range(n_slices):
        slice_2d = binary_volume[i].astype(bool)

        # For 2D slice, calculate perimeter and area
        # Perimeter = edge length in pixels
        # Area = number of pore pixels

        try:
            # Find contours (perimeter)
            contours = measure.find_contours(slice_2d.astype(float), 0.5)

            # Calculate total perimeter
            perimeter_pixels = 0
            for contour in contours:
                # Calculate contour length
                perimeter_pixels += len(contour)

            # Convert to micrometers
            perimeter_um = perimeter_pixels * voxel_size_um

        except:
            # Fallback: count edge pixels
            from scipy import ndimage
            struct = ndimage.generate_binary_structure(2, 1)
            dilated = ndimage.binary_dilation(slice_2d, structure=struct)
            boundary = dilated & ~slice_2d
            perimeter_pixels = np.sum(boundary)
            perimeter_um = perimeter_pixels * voxel_size_um

        # Calculate area
        area_um2 = np.sum(slice_2d) * (voxel_size_um ** 2)

        # SSA for 2D (perimeter per unit area)
        ssa_2d = perimeter_um / area_um2 if area_um2 > 0 else 0
        ssa_per_slice.append(ssa_2d)

    return ssa_per_slice


# ============================================================================
# LOAD DATA
# ============================================================================

perf_tracker.start("Data Loading")

print("\n" + "="*70)
print("LOADING DATA")
print("="*70)

print("\nLoading volumes...")
noisy_volume = imread(config.NOISY_PATH).astype(np.float32)
clean_volume = imread(config.CLEAN_PATH).astype(np.float32)

print(f"\nVolume shape: {noisy_volume.shape}")
print(f"Noisy range: [{noisy_volume.min():.2f}, {noisy_volume.max():.2f}]")
print(f"Clean range: [{clean_volume.min():.2f}, {clean_volume.max():.2f}]")

n_slices = noisy_volume.shape[0]

# Split slices
all_indices = np.arange(n_slices)
train_indices, temp_indices = train_test_split(all_indices, test_size=0.3, random_state=42)
val_indices, test_indices = train_test_split(temp_indices, test_size=0.5, random_state=42)

print(f"\nData split:")
print(f"  Train slices: {len(train_indices)} ({len(train_indices)/n_slices*100:.1f}%)")
print(f"  Val slices: {len(val_indices)} ({len(val_indices)/n_slices*100:.1f}%)")
print(f"  Test slices: {len(test_indices)} ({len(test_indices)/n_slices*100:.1f}%)")

# Create datasets
train_dataset = SliceDataset(noisy_volume, clean_volume, train_indices,
                            config.PATCH_SIZE, config.PATCHES_PER_SLICE)
val_dataset = SliceDataset(noisy_volume, clean_volume, val_indices,
                          config.PATCH_SIZE, config.PATCHES_PER_SLICE)

train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE,
                         shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=config.BATCH_SIZE,
                       shuffle=False, num_workers=2, pin_memory=True)

perf_tracker.end()

# Show hardware state after data loading
hw_monitor.print_stats("After Data Loading")

# ============================================================================
# BUILD MODEL
# ============================================================================

perf_tracker.start("Model Initialization")

print("\n" + "="*70)
print("BUILDING MODEL")
print("="*70)

model = UNet2D(in_ch=1, out_ch=1).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=config.LEARNING_RATE)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min',
                                                 patience=config.LR_PATIENCE,
                                                 factor=config.LR_FACTOR)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}")

perf_tracker.end()

# Show hardware state after model init
hw_monitor.print_stats("After Model Initialization")

perf_tracker.start("Training Phase")

print("\n" + "="*70)
print("TRAINING PHASE")
print("="*70)

best_val_loss = float('inf')
train_losses, val_losses = [], []
learning_rates = []
patience_counter = 0

epoch_times = []

for epoch in range(config.EPOCHS):
    epoch_start = time.time()

    train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss = validate(model, val_loader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    learning_rates.append(optimizer.param_groups[0]['lr'])

    scheduler.step(val_loss)

    epoch_time = time.time() - epoch_start
    epoch_times.append(epoch_time)

    print(f"Epoch {epoch+1:3d}/{config.EPOCHS} - Train: {train_loss:.6f}, Val: {val_loss:.6f}, "
          f"LR: {optimizer.param_groups[0]['lr']:.2e}, Time: {epoch_time:.2f}s")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_2dunet_model.pth')
        print(f"  ✓ Best model saved (Val Loss: {val_loss:.6f})")
        patience_counter = 0
    else:
        patience_counter += 1

    if patience_counter >= config.PATIENCE:
        print(f"\nEarly stopping at epoch {epoch+1}")
        break

training_time = perf_tracker.end()

# Show hardware state after training
hw_monitor.print_stats("After Training")

# Calculate training statistics
avg_epoch_time = np.mean(epoch_times)
total_epochs_run = len(train_losses)

print(f"\n📊 Training Statistics:")
print(f"  Total epochs: {total_epochs_run}")
print(f"  Avg time per epoch: {avg_epoch_time:.2f}s")
print(f"  Total training time: {training_time:.2f}s ({training_time/60:.2f} min)")

# Plot training curves
plt.figure(figsize=(10, 6))
plt.plot(train_losses, label='Training Loss', linewidth=2)
plt.plot(val_losses, label='Validation Loss', linewidth=2)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss (MSE)', fontsize=12)
plt.title('2D U-Net Training Curves', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.savefig('training_curves_2dunet.png', dpi=300, bbox_inches='tight')
plt.show()

# Load best model
model.load_state_dict(torch.load('best_2dunet_model.pth'))

# ============================================================================
# DENOISING
# ============================================================================

perf_tracker.start("Volume Denoising")

print("\n" + "="*70)
print("DENOISING VOLUME")
print("="*70)

denoised_volume = denoise_volume(model, noisy_volume, device)

perf_tracker.end()

# Show hardware state after denoising
hw_monitor.print_stats("After Denoising")

perf_tracker.start("Image Quality Analysis (PSNR, SSIM, MSE)")

print("Standardizing volumes for fair comparison...")
noisy_std, denoised_std, truth_std, data_range = standardize_volumes_for_comparison(
    noisy_volume,
    denoised_volume,
    clean_volume,
    bit_depth=config.TARGET_BIT_DEPTH
)

print("\n" + "="*70)
print("CALCULATING IMAGE QUALITY METRICS")
print("="*70)
print(f"Using standardized {config.TARGET_BIT_DEPTH}-bit volumes")
print(f"Data range: 0 - {data_range}")

metrics_before_all = []
metrics_after_all = []

for slice_idx in range(n_slices):
    # Use standardized volumes for metrics
    noisy_slice = noisy_std[slice_idx]
    denoised_slice = denoised_std[slice_idx]
    truth_slice = truth_std[slice_idx]

    metrics_before = calculate_metrics(truth_slice, noisy_slice, data_range=data_range)
    metrics_after = calculate_metrics(truth_slice, denoised_slice, data_range=data_range)

    metrics_before_all.append(metrics_before)
    metrics_after_all.append(metrics_after)

# Calculate statistics
avg_psnr_before = np.mean([m['PSNR'] for m in metrics_before_all])
avg_psnr_after = np.mean([m['PSNR'] for m in metrics_after_all])
avg_ssim_before = np.mean([m['SSIM'] for m in metrics_before_all])
avg_ssim_after = np.mean([m['SSIM'] for m in metrics_after_all])
avg_mse_before = np.mean([m['MSE'] for m in metrics_before_all])
avg_mse_after = np.mean([m['MSE'] for m in metrics_after_all])

print(f"\nBEFORE Denoising:")
print(f"  PSNR: {avg_psnr_before:.2f} dB")
print(f"  SSIM: {avg_ssim_before:.4f}")

print(f"\nAFTER 2D U-Net:")
print(f"  PSNR: {avg_psnr_after:.2f} dB (+{avg_psnr_after - avg_psnr_before:.2f})")
print(f"  SSIM: {avg_ssim_after:.4f} (+{avg_ssim_after - avg_ssim_before:.4f})")

# Statistical significance
psnr_improvements = [after['PSNR'] - before['PSNR']
                     for before, after in zip(metrics_before_all, metrics_after_all)]
ssim_improvements = [after['SSIM'] - before['SSIM']
                     for before, after in zip(metrics_before_all, metrics_after_all)]

t_stat_psnr, p_value_psnr = stats.ttest_1samp(psnr_improvements, 0)
t_stat_ssim, p_value_ssim = stats.ttest_1samp(ssim_improvements, 0)

print(f"\nStatistical Significance:")
print(f"  PSNR: t={t_stat_psnr:.4f}, p={p_value_psnr:.6f}")
print(f"  SSIM: t={t_stat_ssim:.4f}, p={p_value_ssim:.6f}")

perf_tracker.end()

# ============================================================================
# POROSITY ANALYSIS
# ============================================================================

perf_tracker.start("Porosity Analysis")

print("\n" + "="*70)
print("POROSITY ANALYSIS - ROCK PORE IMAGES")
print("="*70)
print(f"Using standardized {config.TARGET_BIT_DEPTH}-bit volumes for consistent thresholding")

# Step 1: Determine threshold (using standardized ground truth)
print("\nStep 1: Determining optimal threshold method...")
sample_slice_idx = n_slices // 2
sample_slice = truth_std[sample_slice_idx]

# Normalize for threshold calculation
if truth_std.dtype == np.uint8:
    sample_norm = sample_slice.astype(np.float32) / 255.0
elif truth_std.dtype == np.uint16:
    sample_norm = sample_slice.astype(np.float32) / 65535.0
else:
    sample_norm = (sample_slice - sample_slice.min()) / (sample_slice.max() - sample_slice.min() + 1e-8)

otsu_thresh = threshold_otsu(sample_norm)
multiotsu_thresh = threshold_multiotsu(sample_norm, classes=3)

print(f"Otsu threshold: {otsu_thresh:.4f}")
print(f"Multi-Otsu thresholds: {multiotsu_thresh}")

threshold_method = 'otsu'
manual_threshold_value = None

# Step 2: Calculate porosity (using standardized volumes)
print("\n" + "="*70)
print("Step 2: Calculating porosity from standardized volumes...")
print("-" * 70)

print("Analyzing NOISY volume (standardized)...")
porosity_noisy, binary_noisy, thresh_noisy = calculate_porosity(
    noisy_std, method=threshold_method, manual_threshold=manual_threshold_value
)

print("Analyzing DENOISED volume (standardized)...")
porosity_denoised, binary_denoised, thresh_denoised = calculate_porosity(
    denoised_std, method=threshold_method, manual_threshold=manual_threshold_value
)

print("Analyzing GROUND TRUTH volume (standardized)...")
porosity_truth, binary_truth, thresh_truth = calculate_porosity(
    truth_std, method=threshold_method, manual_threshold=manual_threshold_value
)

# Per-slice porosity
porosity_per_slice_noisy = []
porosity_per_slice_denoised = []
porosity_per_slice_truth = []

for slice_idx in range(n_slices):
    por_noisy = np.sum(binary_noisy[slice_idx]) / binary_noisy[slice_idx].size * 100
    por_denoised = np.sum(binary_denoised[slice_idx]) / binary_denoised[slice_idx].size * 100
    por_truth = np.sum(binary_truth[slice_idx]) / binary_truth[slice_idx].size * 100

    porosity_per_slice_noisy.append(por_noisy)
    porosity_per_slice_denoised.append(por_denoised)
    porosity_per_slice_truth.append(por_truth)

# Step 3: Display results
print("\n" + "="*70)
print("POROSITY RESULTS")
print("="*70)

print(f"\nVolume shape: {noisy_volume.shape}")
print(f"Total voxels: {noisy_volume.size:,}")

print(f"\n{'Volume':<25} {'Threshold':<12} {'Porosity (%)':<15}")
print("-" * 70)
print(f"{'Ground Truth':<25} {thresh_truth:<12.4f} {porosity_truth:<15.2f}")
print(f"{'Noisy':<25} {thresh_noisy:<12.4f} {porosity_noisy:<15.2f}")
print(f"{'Denoised (2D U-Net)':<25} {thresh_denoised:<12.4f} {porosity_denoised:<15.2f}")

noisy_error = abs(porosity_noisy - porosity_truth)
denoised_error = abs(porosity_denoised - porosity_truth)
improvement = ((noisy_error - denoised_error) / noisy_error * 100) if noisy_error > 0 else 0

print(f"\n{'Comparison':<25} {'Error vs Truth':<25}")
print("-" * 70)
print(f"{'Noisy Error':<25} {noisy_error:>8.2f}%")
print(f"{'Denoised Error':<25} {denoised_error:>8.2f}%")
print(f"{'Error Reduction':<25} {improvement:>8.2f}%")

perf_tracker.end()

# ============================================================================
# SPECIFIC SURFACE AREA ANALYSIS
# ============================================================================

perf_tracker.start("Specific Surface Area Analysis")

print("\n" + "="*70)
print("SPECIFIC SURFACE AREA (SSA) ANALYSIS")
print("="*70)

print(f"\nUsing parameters:")
print(f"  Voxel size: {config.VOXEL_SIZE_UM} μm")
print(f"  Method: Marching Cubes (3D) + Contour (2D per-slice)")

# Overall SSA for entire volume (3D analysis)
print("\nCalculating overall SSA (3D analysis)...")
print("  This may take a moment for marching cubes algorithm...")

ssa_truth, sa_truth, vol_truth = calculate_specific_surface_area(
    binary_truth, config.VOXEL_SIZE_UM
)
print(f"  ✓ Ground Truth SSA calculated")

ssa_noisy, sa_noisy, vol_noisy = calculate_specific_surface_area(
    binary_noisy, config.VOXEL_SIZE_UM
)
print(f"  ✓ Noisy SSA calculated")

ssa_denoised, sa_denoised, vol_denoised = calculate_specific_surface_area(
    binary_denoised, config.VOXEL_SIZE_UM
)
print(f"  ✓ Denoised SSA calculated")

print(f"\n{'Volume':<25} {'Porosity (%)':<15} {'SSA (μm⁻¹)':<20} {'Surface Area (μm²)':<20}")
print("-" * 85)
print(f"{'Ground Truth':<25} {porosity_truth:<15.2f} {ssa_truth:<20.6f} {sa_truth:<20.2f}")
print(f"{'Noisy':<25} {porosity_noisy:<15.2f} {ssa_noisy:<20.6f} {sa_noisy:<20.2f}")
print(f"{'Denoised (2D U-Net)':<25} {porosity_denoised:<15.2f} {ssa_denoised:<20.6f} {sa_denoised:<20.2f}")

# Errors
ssa_error_noisy = abs(ssa_noisy - ssa_truth)
ssa_error_denoised = abs(ssa_denoised - ssa_truth)
ssa_improvement = ((ssa_error_noisy - ssa_error_denoised) / ssa_error_noisy * 100) if ssa_error_noisy > 0 else 0

print(f"\n{'Comparison':<25} {'SSA Error (μm⁻¹)':<25} {'Error Reduction':<20}")
print("-" * 70)
print(f"{'Noisy Error':<25} {ssa_error_noisy:>8.6f} {'':<20}")
print(f"{'Denoised Error':<25} {ssa_error_denoised:>8.6f} {ssa_improvement:>19.2f}%")

# Per-slice SSA (2D analysis)
print("\nCalculating per-slice SSA (2D analysis)...")
ssa_per_slice_truth = analyze_ssa_per_slice(binary_truth, config.VOXEL_SIZE_UM)
ssa_per_slice_noisy = analyze_ssa_per_slice(binary_noisy, config.VOXEL_SIZE_UM)
ssa_per_slice_denoised = analyze_ssa_per_slice(binary_denoised, config.VOXEL_SIZE_UM)

# Statistical analysis
ssa_error_per_slice_noisy = [abs(n - t) for n, t in zip(ssa_per_slice_noisy, ssa_per_slice_truth)]
ssa_error_per_slice_denoised = [abs(d - t) for d, t in zip(ssa_per_slice_denoised, ssa_per_slice_truth)]

mean_ssa_error_noisy = np.mean(ssa_error_per_slice_noisy)
std_ssa_error_noisy = np.std(ssa_error_per_slice_noisy)
mean_ssa_error_denoised = np.mean(ssa_error_per_slice_denoised)
std_ssa_error_denoised = np.std(ssa_error_per_slice_denoised)

print(f"\nPer-Slice Statistics (2D SSA):")
print(f"  Mean SSA Error (Noisy):    {mean_ssa_error_noisy:.6f} ± {std_ssa_error_noisy:.6f} μm⁻¹")
print(f"  Mean SSA Error (Denoised): {mean_ssa_error_denoised:.6f} ± {std_ssa_error_denoised:.6f} μm⁻¹")

# Paired t-test
t_stat_ssa, p_value_ssa = stats.ttest_rel(ssa_error_per_slice_noisy, ssa_error_per_slice_denoised)

print(f"\nPaired t-test (Noisy vs Denoised SSA errors):")
print(f"  t-statistic: {t_stat_ssa:.4f}")
print(f"  p-value: {p_value_ssa:.6f}")

if p_value_ssa < 0.001:
    print(f"  ✓ SSA error reduction is HIGHLY SIGNIFICANT (p < 0.001)")
elif p_value_ssa < 0.05:
    print(f"  ✓ SSA error reduction is SIGNIFICANT (p < 0.05)")
else:
    print(f"  ✗ SSA error reduction is NOT significant (p >= 0.05)")

# Correlation
corr_ssa_noisy = np.corrcoef(ssa_per_slice_noisy, ssa_per_slice_truth)[0, 1]
corr_ssa_denoised = np.corrcoef(ssa_per_slice_denoised, ssa_per_slice_truth)[0, 1]

print(f"\nCorrelation with Ground Truth:")
print(f"  Noisy SSA:    r = {corr_ssa_noisy:.4f}")
print(f"  Denoised SSA: r = {corr_ssa_denoised:.4f}")
print(f"  Improvement: {(corr_ssa_denoised - corr_ssa_noisy):.4f}")

perf_tracker.end()

# ============================================================================
# VISUALIZATIONS
# ============================================================================

# Plot Specific Surface Area per slice
fig, axes = plt.subplots(2, 1, figsize=(16, 10))

slices = range(n_slices)

# SSA values
axes[0].plot(slices, ssa_per_slice_truth, label='Ground Truth',
            linewidth=2.5, alpha=0.9, color='green')
axes[0].plot(slices, ssa_per_slice_noisy, label='Noisy',
            linewidth=2, alpha=0.7, linestyle='--', color='red')
axes[0].plot(slices, ssa_per_slice_denoised, label='Denoised (2D U-Net)',
            linewidth=2, alpha=0.7, color='blue')
axes[0].axhline(y=np.mean(ssa_per_slice_truth), color='green', linestyle=':', alpha=0.5,
               label=f'Avg Truth: {np.mean(ssa_per_slice_truth):.6f} μm⁻¹')
axes[0].axhline(y=np.mean(ssa_per_slice_denoised), color='blue', linestyle=':', alpha=0.5,
               label=f'Avg Denoised: {np.mean(ssa_per_slice_denoised):.6f} μm⁻¹')
axes[0].set_xlabel('Slice Index', fontsize=12)
axes[0].set_ylabel('Specific Surface Area (μm⁻¹)', fontsize=12)
axes[0].set_title('Specific Surface Area per Slice (2D U-Net)', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=10, loc='best')
axes[0].grid(True, alpha=0.3)

# SSA errors
axes[1].plot(slices, ssa_error_per_slice_noisy, label='Noisy Error',
            linewidth=2, alpha=0.7, color='red')
axes[1].plot(slices, ssa_error_per_slice_denoised, label='Denoised Error',
            linewidth=2, alpha=0.7, color='blue')
axes[1].fill_between(slices, ssa_error_per_slice_noisy, ssa_error_per_slice_denoised,
                     alpha=0.2, color='green', label='Improvement Region')
axes[1].axhline(y=mean_ssa_error_noisy, color='red', linestyle='--', alpha=0.5,
               label=f'Avg Noisy Error: {mean_ssa_error_noisy:.6f} μm⁻¹')
axes[1].axhline(y=mean_ssa_error_denoised, color='blue', linestyle='--', alpha=0.5,
               label=f'Avg Denoised Error: {mean_ssa_error_denoised:.6f} μm⁻¹')
axes[1].set_xlabel('Slice Index', fontsize=12)
axes[1].set_ylabel('Absolute SSA Error (μm⁻¹)', fontsize=12)
axes[1].set_title('Specific Surface Area Error per Slice (2D U-Net)', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=10, loc='best')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('specific_surface_area_analysis_2dunet.png', dpi=300, bbox_inches='tight')
plt.show()

# ============================================================================
# EXPORT TO CSV
# ============================================================================

print("\n" + "="*70)
print("EXPORTING ALL RESULTS TO CSV FILES")
print("="*70)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# CSV 1: Combined per-slice analysis (WITH SSA)
df_combined = pd.DataFrame({
    'Slice_Index': range(n_slices),
    'PSNR_Before_dB': [m['PSNR'] for m in metrics_before_all],
    'PSNR_After_dB': [m['PSNR'] for m in metrics_after_all],
    'PSNR_Improvement_dB': [after['PSNR'] - before['PSNR']
                            for before, after in zip(metrics_before_all, metrics_after_all)],
    'SSIM_Before': [m['SSIM'] for m in metrics_before_all],
    'SSIM_After': [m['SSIM'] for m in metrics_after_all],
    'SSIM_Improvement': [after['SSIM'] - before['SSIM']
                         for before, after in zip(metrics_before_all, metrics_after_all)],
    'MSE_Before': [m['MSE'] for m in metrics_before_all],
    'MSE_After': [m['MSE'] for m in metrics_after_all],
    'Porosity_Ground_Truth_%': porosity_per_slice_truth,
    'Porosity_Noisy_%': porosity_per_slice_noisy,
    'Porosity_Denoised_%': porosity_per_slice_denoised,
    'Porosity_Error_Noisy_%': [abs(n - t) for n, t in zip(porosity_per_slice_noisy, porosity_per_slice_truth)],
    'Porosity_Error_Denoised_%': [abs(d - t) for d, t in zip(porosity_per_slice_denoised, porosity_per_slice_truth)],
    'SSA_Truth_um_inv': ssa_per_slice_truth,
    'SSA_Noisy_um_inv': ssa_per_slice_noisy,
    'SSA_Denoised_um_inv': ssa_per_slice_denoised,
    'SSA_Error_Noisy_um_inv': ssa_error_per_slice_noisy,
    'SSA_Error_Denoised_um_inv': ssa_error_per_slice_denoised,
})

csv1 = f'combined_analysis_with_SSA_2dunet_{timestamp}.csv'
df_combined.to_csv(csv1, index=False, float_format='%.8f')
print(f"✓ Saved: {csv1}")

# CSV 2: Overall summary (WITH PERFORMANCE & SSA & HARDWARE)
perf_summary = perf_tracker.get_summary()
peak_memory = perf_tracker.get_peak_memory_usage()

summary_data = {
    'Metric': [
        # Dataset info
        'Volume_Depth', 'Volume_Height', 'Volume_Width', 'Total_Voxels',

        # Bit depth standardization
        'Original_Noisy_dtype', 'Original_Denoised_dtype', 'Original_Truth_dtype',
        'Standardized_Bit_Depth', 'Standardized_dtype', 'Data_Range_For_Metrics',

        # Threshold
        'Threshold_Method', 'Threshold_Ground_Truth', 'Threshold_Noisy', 'Threshold_Denoised',

        # Porosity
        'Porosity_Ground_Truth_%', 'Porosity_Noisy_%', 'Porosity_Denoised_%',
        'Porosity_Error_Noisy_%', 'Porosity_Error_Denoised_%', 'Porosity_Error_Reduction_%',

        # Specific Surface Area
        'Voxel_Size_um',
        'SSA_3D_Truth_um_inv', 'SSA_3D_Noisy_um_inv', 'SSA_3D_Denoised_um_inv',
        'SSA_3D_Error_Noisy_um_inv', 'SSA_3D_Error_Denoised_um_inv', 'SSA_3D_Error_Reduction_%',
        'Surface_Area_Truth_um2', 'Surface_Area_Noisy_um2', 'Surface_Area_Denoised_um2',
        'SSA_2D_Mean_Truth_um_inv', 'SSA_2D_Mean_Noisy_um_inv', 'SSA_2D_Mean_Denoised_um_inv',

        # Image quality
        'PSNR_Before_Mean_dB', 'PSNR_After_Mean_dB', 'PSNR_Improvement_dB',
        'SSIM_Before_Mean', 'SSIM_After_Mean', 'SSIM_Improvement',
        'MSE_Before_Mean', 'MSE_After_Mean',

        # Statistical tests
        'PSNR_t_statistic', 'PSNR_p_value',
        'SSIM_t_statistic', 'SSIM_p_value',
        'Porosity_t_statistic', 'Porosity_p_value',
        'SSA_t_statistic', 'SSA_p_value',

        # Correlations
        'Porosity_Correlation_Noisy', 'Porosity_Correlation_Denoised',
        'SSA_Correlation_Noisy', 'SSA_Correlation_Denoised',

        # Model info
        'Model_Total_Parameters', 'Model_Architecture',
        'Training_Epochs', 'Best_Validation_Loss', 'Avg_Epoch_Time_s',

        # Performance times
        'Time_Data_Loading_s', 'Time_Model_Init_s', 'Time_Training_s',
        'Time_Denoising_s', 'Time_Image_Quality_Analysis_s',
        'Time_Porosity_Analysis_s', 'Time_SSA_Analysis_s',
        'Time_Total_s',

        # Hardware - Peak Usage
        'Peak_GPU_Memory_GB', 'Peak_RAM_Usage_GB', 'Peak_Process_RAM_GB',
        'System_Total_RAM_GB', 'GPU_Name', 'CPU_Count'
    ],
    'Value': [
        # Dataset
        noisy_volume.shape[0], noisy_volume.shape[1], noisy_volume.shape[2], noisy_volume.size,

        # Bit depth info
        str(noisy_volume.dtype), str(denoised_volume.dtype), str(clean_volume.dtype),
        config.TARGET_BIT_DEPTH, str(noisy_std.dtype), data_range,

        # Threshold
        threshold_method, thresh_truth, thresh_noisy, thresh_denoised,

        # Porosity
        porosity_truth, porosity_noisy, porosity_denoised,
        noisy_error, denoised_error, improvement,

        # Specific Surface Area
        config.VOXEL_SIZE_UM,
        ssa_truth, ssa_noisy, ssa_denoised,
        ssa_error_noisy, ssa_error_denoised, ssa_improvement,
        sa_truth, sa_noisy, sa_denoised,
        np.mean(ssa_per_slice_truth), np.mean(ssa_per_slice_noisy), np.mean(ssa_per_slice_denoised),

        # Image quality
        avg_psnr_before, avg_psnr_after, avg_psnr_after - avg_psnr_before,
        avg_ssim_before, avg_ssim_after, avg_ssim_after - avg_ssim_before,
        avg_mse_before, avg_mse_after,

        # Stats
        t_stat_psnr, p_value_psnr,
        t_stat_ssim, p_value_ssim,
        stats.ttest_rel([abs(n - t) for n, t in zip(porosity_per_slice_noisy, porosity_per_slice_truth)],
                       [abs(d - t) for d, t in zip(porosity_per_slice_denoised, porosity_per_slice_truth)])[0],
        stats.ttest_rel([abs(n - t) for n, t in zip(porosity_per_slice_noisy, porosity_per_slice_truth)],
                       [abs(d - t) for d, t in zip(porosity_per_slice_denoised, porosity_per_slice_truth)])[1],
        t_stat_ssa, p_value_ssa,

        # Correlations
        corr_noisy_truth := np.corrcoef(porosity_per_slice_noisy, porosity_per_slice_truth)[0, 1],
        corr_denoised_truth := np.corrcoef(porosity_per_slice_denoised, porosity_per_slice_truth)[0, 1],
        corr_ssa_noisy, corr_ssa_denoised,

        # Model
        total_params, '2D U-Net',
        total_epochs_run, best_val_loss, avg_epoch_time,

        # Performance
        perf_summary['individual'].get('Data Loading', 0),
        perf_summary['individual'].get('Model Initialization', 0),
        perf_summary['individual'].get('Training Phase', 0),
        perf_summary['individual'].get('Volume Denoising', 0),
        perf_summary['individual'].get('Image Quality Analysis (PSNR, SSIM, MSE)', 0),
        perf_summary['individual'].get('Porosity Analysis', 0),
        perf_summary['individual'].get('Specific Surface Area Analysis', 0),
        perf_summary['total'],

        # Hardware
        peak_memory['peak_gpu_gb'],
        peak_memory['peak_ram_gb'],
        peak_memory['peak_process_ram_gb'],
        psutil.virtual_memory().total / (1024**3),
        torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU Only',
        psutil.cpu_count()
    ]
}

df_summary = pd.DataFrame(summary_data)
csv2 = f'overall_summary_with_performance_2dunet_{timestamp}.csv'
df_summary.to_csv(csv2, index=False, float_format='%.8f')
print(f"✓ Saved: {csv2}")

# CSV 3: Performance breakdown with hardware
hardware_per_phase = []
for phase in perf_summary['individual'].keys():
    phase_data = {
        'Phase': phase,
        'Time_seconds': perf_summary['individual'][phase],
        'Time_formatted': perf_summary['formatted'][phase],
        'Percentage': perf_summary['individual'][phase]/perf_summary['total']*100
    }

    # Add hardware stats if available
    if phase in perf_summary['hardware']:
        hw_stats = perf_summary['hardware'][phase]
        if 'peak_gpu_gb' in hw_stats:
            phase_data['Peak_GPU_Memory_GB'] = hw_stats['peak_gpu_gb']
        if 'start' in hw_stats and 'end' in hw_stats:
            phase_data['RAM_Start_GB'] = hw_stats['start']['process_ram_gb']
            phase_data['RAM_End_GB'] = hw_stats['end']['process_ram_gb']
            phase_data['RAM_Delta_GB'] = hw_stats['end']['process_ram_gb'] - hw_stats['start']['process_ram_gb']

    hardware_per_phase.append(phase_data)

# Add total row
hardware_per_phase.append({
    'Phase': 'TOTAL',
    'Time_seconds': perf_summary['total'],
    'Time_formatted': perf_summary['total_formatted'],
    'Percentage': 100.0,
    'Peak_GPU_Memory_GB': peak_memory['peak_gpu_gb'],
    'RAM_Start_GB': '',
    'RAM_End_GB': peak_memory['peak_process_ram_gb'],
    'RAM_Delta_GB': ''
})

df_performance = pd.DataFrame(hardware_per_phase)

csv3 = f'performance_breakdown_with_hardware_2dunet_{timestamp}.csv'
df_performance.to_csv(csv3, index=False, float_format='%.4f')
print(f"✓ Saved: {csv3}")

# ============================================================================
# PRINT FINAL PERFORMANCE SUMMARY
# ============================================================================

perf_tracker.print_summary()

# Print final hardware state
hw_monitor.print_stats("Final")

print("\n" + "="*70)
print("HARDWARE UTILIZATION SUMMARY")
print("="*70)
print(f"  Peak GPU Memory:     {peak_memory['peak_gpu_gb']:.2f} GB")
print(f"  Peak RAM Usage:      {peak_memory['peak_ram_gb']:.2f} GB")
print(f"  Peak Process RAM:    {peak_memory['peak_process_ram_gb']:.2f} GB")
print(f"  System Total RAM:    {psutil.virtual_memory().total / (1024**3):.2f} GB")
if torch.cuda.is_available():
    print(f"  GPU Device:          {torch.cuda.get_device_name(0)}")
    gpu_total = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"  GPU Total Memory:    {gpu_total:.2f} GB")
    print(f"  GPU Utilization:     {(peak_memory['peak_gpu_gb']/gpu_total)*100:.1f}%")
print(f"  CPU Count:           {psutil.cpu_count()}")

print("\n" + "="*70)
print("2D U-NET COMPLETE ANALYSIS FINISHED")
print("="*70)

print(f"\n📊 FINAL SUMMARY:")
print(f"  Architecture: 2D U-Net")
print(f"  Parameters: {total_params:,}")
print(f"  Total Time: {perf_summary['total_formatted']}")
print(f"  Peak GPU: {peak_memory['peak_gpu_gb']:.2f} GB")
print(f"  Peak RAM: {peak_memory['peak_process_ram_gb']:.2f} GB")

print(f"\n📈 IMAGE QUALITY:")
print(f"  PSNR: {avg_psnr_before:.2f} → {avg_psnr_after:.2f} dB (+{avg_psnr_after - avg_psnr_before:.2f})")
print(f"  SSIM: {avg_ssim_before:.4f} → {avg_ssim_after:.4f} (+{avg_ssim_after - avg_ssim_before:.4f})")

print(f"\n🔬 POROSITY:")
print(f"  Ground Truth: {porosity_truth:.2f}%")
print(f"  Error Reduction: {improvement:.1f}%")

print(f"\n📐 SPECIFIC SURFACE AREA:")
print(f"  Ground Truth (3D): {ssa_truth:.6f} μm⁻¹")
print(f"  Noisy Error: {ssa_error_noisy:.6f} μm⁻¹")
print(f"  Denoised Error: {ssa_error_denoised:.6f} μm⁻¹")
print(f"  Improvement: {ssa_improvement:.1f}%")

print(f"\n💾 OUTPUT FILES:")
print(f"  • Model: best_2dunet_model.pth")
print(f"  • Original denoised: denoised_volume_original_2dunet.tif")
print(f"  • Standardized volumes ({config.TARGET_BIT_DEPTH}-bit): standardized_*_2dunet.tif (3 files)")
print(f"  • Binary segmentation: binary_*_2dunet.tif (3 files)")
print(f"  • CSV files: 3 comprehensive analysis files (with hardware metrics)")
print(f"  • Visualizations: training curves, SSA analysis plots")

print("\n✅ All analyses complete with:")
print(f"  ✓ Hardware monitoring (GPU + RAM)")
print(f"  ✓ Apple-to-apple comparison ({config.TARGET_BIT_DEPTH}-bit standardization)")
print(f"  ✓ Porosity analysis")
print(f"  ✓ Specific Surface Area analysis")
print(f"  ✓ Complete performance tracking")

In [ ]:
from tifffile import imwrite
import os

print("Menyimpan file output...")

# 1. Simpan hasil denoising murni (float32)
if 'denoised_volume' in locals():
    save_path = 'denoised_volume_final.tif'
    imwrite(save_path, denoised_volume)
    print(f"✅ Berhasil menyimpan: {save_path}")
    print(f"   Size: {os.path.getsize(save_path)/1e6:.2f} MB")
else:
    print("⚠️ Variabel 'denoised_volume' tidak ditemukan di memori.")

# 2. Simpan versi visualisasi (uint8 - standar 0-255)
# Ini lebih mudah dibuka di image viewer biasa
if 'denoised_std' in locals():
    save_path_vis = 'denoised_volume_visual_8bit.tif'
    imwrite(save_path_vis, denoised_std)
    print(f"✅ Berhasil menyimpan versi visual (8-bit): {save_path_vis}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

def calculate_dice_score(pred_binary, true_binary):
    """Menghitung Dice Coefficient (Overlap accuracy)"""
    intersection = np.logical_and(pred_binary, true_binary)
    dice = 2. * intersection.sum() / (pred_binary.sum() + true_binary.sum())
    return dice

# --- 1. VISUALISASI RESIDUAL MAP (SLICE TENGAH) ---
mid_slice = noisy_std.shape[0] // 2

# Ambil slice tengah
n_slice = noisy_std[mid_slice]
d_slice = denoised_std[mid_slice]
t_slice = truth_std[mid_slice]

# Hitung Residual (Apa yang dibuang oleh model?)
# Kita tambah 128 (setengah gray) supaya noise terlihat abu-abu, bukan hitam
noise_residual = (n_slice.astype(float) - d_slice.astype(float))
error_map = np.abs(t_slice.astype(float) - d_slice.astype(float))

plt.figure(figsize=(20, 10))

# Baris 1: Gambar Asli
plt.subplot(2, 4, 1); plt.imshow(n_slice, cmap='gray'); plt.title(f"Noisy (Slice {mid_slice})"); plt.axis('off')
plt.subplot(2, 4, 2); plt.imshow(d_slice, cmap='gray'); plt.title("Denoised (2D U-Net)"); plt.axis('off')
plt.subplot(2, 4, 3); plt.imshow(t_slice, cmap='gray'); plt.title("Ground Truth"); plt.axis('off')

# Baris 1 Kolom 4: Histogram Comparison
plt.subplot(2, 4, 4)
plt.hist(t_slice.ravel(), bins=50, alpha=0.5, label='Truth', color='green', density=True)
plt.hist(d_slice.ravel(), bins=50, alpha=0.5, label='Denoised', color='blue', density=True)
plt.title("Histogram Intensitas")
plt.legend()
plt.grid(True, alpha=0.3)

# Baris 2: Analisis Detail
plt.subplot(2, 4, 5)
# Residual map: Harusnya terlihat seperti noise acak. Kalau ada struktur batuan, berarti over-smooth.
plt.imshow(noise_residual, cmap='coolwarm', vmin=-np.std(noise_residual)*3, vmax=np.std(noise_residual)*3)
plt.title("Residual (Noisy - Denoised)\n(Harus acak/noise only)")
plt.colorbar(fraction=0.046, pad=0.04)
plt.axis('off')

plt.subplot(2, 4, 6)
plt.imshow(error_map, cmap='magma')
plt.title("Absolute Error Map\n(Semakin gelap semakin akurat)")
plt.colorbar(fraction=0.046, pad=0.04)
plt.axis('off')

# Baris 2 Kolom 7 & 8: Binary Overlay (Penting untuk Porositas)
# Binerisasi dengan threshold Otsu yang sudah dihitung sebelumnya
bin_d = d_slice < thresh_denoised # Pori = True
bin_t = t_slice < thresh_truth    # Pori = True

# Overlay: Hijau=Truth, Merah=Prediksi, Kuning=Overlap (Benar)
overlay = np.zeros((bin_d.shape[0], bin_d.shape[1], 3), dtype=float)
overlay[..., 0] = bin_d  # Channel Merah (Prediksi)
overlay[..., 1] = bin_t  # Channel Hijau (Truth)
# Jika Merah + Hijau bertemu jadi Kuning (Overlap Benar)

plt.subplot(2, 4, 7)
plt.imshow(overlay)
plt.title("Pore Segmentation Overlay\n(Kuning=Benar, Merah=False Pos, Hijau=Miss)")
plt.axis('off')

# Teks Metrik Tambahan
dice_score = calculate_dice_score(bin_d, bin_t)
iou_score = dice_score / (2 - dice_score)

plt.subplot(2, 4, 8)
plt.text(0.1, 0.6, "METRIK SEGMENTASI:\n(Akurasi Posisi Pori)", fontsize=12, fontweight='bold')
plt.text(0.1, 0.4, f"Dice Score: {dice_score:.4f}\n(Target > 0.90)", fontsize=12)
plt.text(0.1, 0.2, f"IoU Score:  {iou_score:.4f}", fontsize=12)
plt.axis('off')

plt.tight_layout()
plt.savefig('essential_analysis_report.png', dpi=300)
plt.show()

print(f"\nANALISIS ESENSIAL TAMBAHAN:")
print(f"1. Dice Score (Overlap Accuracy): {dice_score:.4f}")
print(f"   (Nilai 1.0 berarti posisi pori sempurna sama persis)")
print(f"2. Residual Map disimpan di 'essential_analysis_report.png'")